# 02 — Tool Calling

This notebook covers defining tools, connecting them to Gemini via LangChain, and letting the model choose which tool to call.

**Kernel:** `ai-engineering`

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_openai import ChatChatOpenAI

model = ChatChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. Defining Tools

Tools are Python functions decorated with `@tool`. The docstring is critical — the model uses it to decide when to call the tool.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Supports basic arithmetic (+, -, *, /, **, %)."""
    try:
        # Safe evaluation for math expressions only
        allowed = set("0123456789+-*/.()% **")
        if not all(c in allowed for c in expression):
            return "Error: expression contains disallowed characters"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Returns temperature and conditions."""
    # Simulated weather data — replace with real API in production
    weather_data = {
        "tokyo": "18°C, partly cloudy",
        "london": "12°C, overcast with light rain",
        "new york": "22°C, sunny",
        "paris": "15°C, clear skies",
        "sydney": "25°C, sunny"
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return f"Weather in {city}: {weather_data[city_lower]}"
    return f"Weather data not available for {city}. Available cities: {', '.join(weather_data.keys())}"

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another."""
    # Simulated exchange rates
    rates = {
        "USD": 1.0,
        "EUR": 0.92,
        "GBP": 0.79,
        "JPY": 149.5,
        "AUD": 1.53
    }
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()
    
    if from_currency not in rates or to_currency not in rates:
        return f"Unknown currency. Supported: {', '.join(rates.keys())}"
    
    usd_amount = amount / rates[from_currency]
    converted = usd_amount * rates[to_currency]
    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculate Body Mass Index (BMI) from weight in kg and height in meters."""
    if height_m <= 0:
        return "Error: height must be positive"
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI: {bmi:.1f} ({category})"

print("Tools defined:", [t.name for t in [calculate, get_weather, convert_currency, calculate_bmi]])

## 2. Binding Tools to the Model

Use `model.bind_tools()` to tell the model what tools are available. The model will generate tool calls when appropriate.

In [ ]:
tools = [calculate, get_weather, convert_currency, calculate_bmi]
model_with_tools = model.bind_tools(tools)

## 3. Tool Selection by the Model

The model decides which tool to call based on the user's query and the tool docstrings.

In [ ]:
# Test: mathematical question → should call calculate
response = model_with_tools.invoke("What is 15 * 23 + 7?")
print("Response content:", response.content)
print("Tool calls:", response.tool_calls)

In [ ]:
# Test: weather question → should call get_weather
response = model_with_tools.invoke("What's the weather like in Tokyo right now?")
print("Response content:", response.content)
print("Tool calls:", response.tool_calls)

In [ ]:
# Test: currency conversion → should call convert_currency
response = model_with_tools.invoke("Convert 100 USD to EUR")
print("Response content:", response.content)
print("Tool calls:", response.tool_calls)

In [ ]:
# Test: BMI calculation → should call calculate_bmi
response = model_with_tools.invoke("My weight is 70kg and my height is 1.75m. What's my BMI?")
print("Response content:", response.content)
print("Tool calls:", response.tool_calls)

In [ ]:
# Test: general question → should NOT call any tool
response = model_with_tools.invoke("What is the meaning of life?")
print("Response content:", response.content)
print("Tool calls:", response.tool_calls)

## 4. Manual Tool Execution Loop

Understand the tool calling flow by executing it manually:

1. Send user message to model
2. Model returns tool calls
3. Execute tools
4. Send tool results back to model
5. Model generates final response

In [ ]:
def run_with_tools(query: str) -> str:
    """Execute a query with automatic tool calling."""
    messages = [HumanMessage(content=query)]
    
    # Step 1: Get model response (may include tool calls)
    response = model_with_tools.invoke(messages)
    print(f"Model response (before tools): {response.content}")
    print(f"Tool calls: {response.tool_calls}")
    
    # Step 2: If there are tool calls, execute them
    if response.tool_calls:
        messages.append(response)  # Add the assistant's tool call message
        
        # Create tool lookup
        tool_map = {t.name: t for t in tools}
        
        # Execute each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            print(f"\nExecuting: {tool_name}({tool_args})")
            result = tool_map[tool_name].invoke(tool_args)
            print(f"Result: {result}")
            
            # Add tool result to messages
            messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))
        
        # Step 3: Get final response with tool results
        final_response = model_with_tools.invoke(messages)
        return final_response.content
    
    return response.content

In [ ]:
result = run_with_tools("What is 42 * 17?")
print(f"\nFinal answer: {result}")

In [ ]:
result = run_with_tools("What's the weather in London and what is 100 USD in JPY?")
print(f"\nFinal answer: {result}")

## 5. Using LangChain's ToolNode (Automated Execution)

LangChain provides `ToolNode` for automated tool execution within LangGraph workflows.

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

# Simulate a tool call message
response = model_with_tools.invoke("Calculate 25 * 4")

if response.tool_calls:
    # ToolNode handles execution automatically
    tool_result = tool_node.invoke({"messages": [response]})
    print("ToolNode result:")
    for msg in tool_result["messages"]:
        print(f"  {msg.content}")

## 6. Exercise — Build Your Own Tool

Create a tool that takes a list of numbers and returns statistics: mean, median, min, max, and standard deviation.

In [ ]:
import statistics

@tool
def analyze_numbers(numbers: str) -> str:
    """Analyze a comma-separated list of numbers. Returns mean, median, min, max, and standard deviation."""
    # Parse the input
    # Calculate statistics
    # Return formatted result
    pass

# Test it
# result = analyze_numbers.invoke({"numbers": "1, 2, 3, 4, 5, 6, 7, 8, 9, 10"})
# print(result)

## 7. Exercise — Multi-Tool Query

Write a function that handles queries requiring multiple tool calls and prints each step.

In [ ]:
def handle_complex_query(query: str):
    """Handle a query that may require multiple tool calls.
    
    Print each step:
    1. The user query
    2. Which tools the model wants to call
    3. Each tool call and its result
    4. The final response
    """
    pass

# Test with: "What's the weather in Paris and how much is 50 GBP in USD?"
# handle_complex_query("What's the weather in Paris and how much is 50 GBP in USD?")

## Key Takeaways

| Concept | What It Does |
|---------|-------------|
| `@tool` | Decorator to define a tool from a Python function |
| `model.bind_tools(tools)` | Make tools available to the model |
| `response.tool_calls` | List of tool calls the model wants to make |
| `ToolMessage` | Wraps a tool's result to send back to the model |
| `ToolNode` | Automated tool execution for LangGraph |

Tool calling is the foundation for agents. Next: [03-langgraph-workflow.ipynb](03-langgraph-workflow.ipynb) — building stateful agent workflows.